[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_07_gans_generative_adversarial_networks/aula_07_dcgan_cifar10_treinamento.ipynb)

# 🎨 Aula 07 — Caderno Prático 1: Implementação e Treinamento de uma DCGAN do Zero em PyTorch
### **Faculdade Infnet — Pós-Graduação em Visão Computacional com CNNs e Transformers**
**Professor:** Dr. Allan Spadini  
**Metodologia Pedagógica Central:** Situação-Problema do Mundo Real ➔ Solução de Engenharia ➔ Teoria Rigorosa

---

### 🎯 Objetivos Deste Laboratório:
1. **Compreender a Dinâmica Minimax de Soma Zero:** Estruturar a competição adversarial entre um Gerador ($G$) e um Discriminador ($D$).
2. **Implementar a Arquitetura DCGAN (Radford et al., 2015):** Projetar as redes convolucionais profundas com convoluções com stride, convoluções transpostas (`ConvTranspose2d`) e normalização por lote (`BatchNorm2d`).
3. **Rastrear Tensores em PyTorch:** Acompanhar a expansão dimensional $[B, 100, 1, 1] 	o [B, 3, 32, 32]$ e a compressão hierárquica correspondente no discriminador.
4. **Dominar a Heurística Não-Saturante ($-\log D$):** Superar o desvanecimento precoce de gradiente da formulação original de Goodfellow.
5. **Utilizar o Dataset CIFAR-10 RGB ($32\times 32$):** Superar datasets simplistas/batidos (como MNIST ou Fashion-MNIST), treinando a rede em imagens coloridas de três canais do mundo real.
6. **Desacoplamento Computacional com `.detach()`:** Controlar o grafo computacional para atualizar os pesos de $D$ e $G$ isoladamente sem estouro de memória.
7. **Interpolação no Espaço Latente ($z_A \to z_B$):** Explorar a continuidade da variedade geométrica aprendida pelo gerador.



---
## ⚙️ Parte 1: Setup do Ambiente, Reprodutibilidade e Detecção de GPU

Nesta etapa inicial, garantimos reprodutibilidade científica fixando as sementes randômicas de todas as bibliotecas e inspecionamos o acelerador CUDA disponível no ambiente Google Colab.



In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.utils as vutils

# 1. Reprodutibilidade Científica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Detecção Automática de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Dispositivo Ativo: {device}")
if torch.cuda.is_available():
    print(f"🚀 GPU Detectada: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"💾 VRAM Total Disponível: {vram_gb:.2f} GB")
else:
    print("⚠️ Atenção: Executando em CPU. Para acelerar o treino, ative a GPU T4 em: Ambiente de Execução > Alterar tipo de ambiente de execução.")



---
## 📦 Parte 2: Pipeline de Dados com CIFAR-10 RGB ($32\times 32$)

### Metodologia: O Problema da Normalização em GANs
O gerador da DCGAN utiliza como função de ativação na camada de saída a tangente hiperbólica (`Tanh`), cujos valores pertencem estritamente ao intervalo $[-1, 1]$.
Se alimentássemos o discriminador com imagens no intervalo clássico $[0, 1]$, haveria uma assimetria imediata de distribuição que desestabilizaria o jogo minimax.

**Solução de Engenharia:**
Normalizamos as imagens reais do CIFAR-10 com média $\mu = (0.5, 0.5, 0.5)$ e desvio-padrão $\sigma = (0.5, 0.5, 0.5)$:
$$x_{\text{norm}} = \frac{x - 0.5}{0.5} \in [-1, 1]$$

Além disso, para acelerar a convergência didática em GPU T4 e permitir a visualização de morfologias nítidas em poucas épocas, oferecemos a opção de filtrar o dataset para uma classe visualmente coerente (por exemplo, classe **6: 'frog' / sapos** ou **1: 'automobile' / carros**, o clássico benchmark histórico de Radford et al., 2015 e Ian Goodfellow), ou treinar em todas as classes simultaneamente.



In [ ]:
# Configurações do Dataset
IMAGE_SIZE = 32
BATCH_SIZE = 128
CHANNELS = 3

# Transformação com normalização em [-1, 1] para casar com a Tanh do Gerador
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download automático do CIFAR-10 via TorchVision
dataset_raw = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

# Configuração Didática: Filtro por Classe Específica para Convergência Ultrarrápida
# Classe 6 = Sapo ('frog') ou Classe 1 = Carro ('automobile')
FILTER_CLASS = 6  # Altere para None se desejar treinar em todas as 50.000 imagens
CIFAR_CLASSES = dataset_raw.classes

if FILTER_CLASS is not None:
    class_name = CIFAR_CLASSES[FILTER_CLASS]
    indices = [i for i, (_, label) in enumerate(dataset_raw) if label == FILTER_CLASS]
    train_dataset = Subset(dataset_raw, indices)
    print(f"🎯 Treinando DCGAN na Classe Específica: '{class_name.upper()}' ({len(train_dataset)} imagens)")
else:
    train_dataset = dataset_raw
    print(f"🎯 Treinando DCGAN em Todas as 10 Classes do CIFAR-10 ({len(train_dataset)} imagens)")

# DataLoader otimizado
dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"📦 Total de Mini-batches por Época: {len(dataloader)}")



In [ ]:
# Função para desnormalizar e exibir mini-batch de imagens reais
def denormalize(tensor):
    """Converte tensores de [-1, 1] para [0, 1] para plotagem no Matplotlib."""
    return tensor * 0.5 + 0.5

# Visualizar uma amostra de imagens reais do DataLoader
real_batch = next(iter(dataloader))
plt.figure(figsize=(10, 5))
plt.axis("off")
plt.title(f"Amostras Reais de Treinamento (CIFAR-10: {CIFAR_CLASSES[FILTER_CLASS] if FILTER_CLASS is not None else 'Geral'})", fontsize=12, fontweight='bold')
grid_img = vutils.make_grid(denormalize(real_batch[0][:32]), nrow=8, padding=2, normalize=False)
plt.imshow(np.transpose(grid_img.cpu().numpy(), (1, 2, 0)))
plt.show()



---
## 🏗️ Parte 3: A Arquitetura DCGAN do Zero em PyTorch

A **DCGAN** (Radford et al., ICLR 2016) revolucionou a estabilidade de redes adversariais estabelecendo diretrizes arquiteturais estritas:

1. **Substituição de Pooling por Convoluções com Stride:**
   - O Discriminador utiliza convoluções com `stride=2` para reduzir a resolução espacial, permitindo que a rede aprenda seu próprio downsampling.
   - O Gerador utiliza convoluções transpostas fracionadas (`ConvTranspose2d`) com `stride=2` para upsampling.
2. **Normalização por Lote (`BatchNorm2d`):**
   - Estabiliza o fluxo de gradientes e impede que os parâmetros colapsem em um único modo durante o treino. O primeiro estágio do discriminador e a saída final do gerador dispensam BatchNorm para manter o contraste original da imagem.
3. **Funções de Ativação:**
   - **Gerador:** `ReLU` em todas as camadas intermediárias e `Tanh` na camada de saída.
   - **Discriminador:** `LeakyReLU` com inclinação negativa de $0.2$ em todas as camadas para garantir que gradientes fluam mesmo para ativações negativas.
4. **Inicialização de Pesos (`weights_init`):**
   - Radford et al. comprovaram empiricamente que inicializar os pesos a partir de uma distribuição normal $\mathcal{N}(0, 0.02^2)$ é fundamental para prevenir divergência precoce.



In [ ]:
# Hiperparâmetros Arquiteturais da DCGAN
NZ = 100    # Tamanho do vetor latente de ruído z
NGF = 64    # Fator base de canais do Gerador (64 -> 128 -> 256)
NDF = 64    # Fator base de canais do Discriminador (64 -> 128 -> 256)
NC = 3      # Número de canais de cor da imagem (RGB)

class Generator(nn.Module):
    """
    Gerador DCGAN para imagens 32x32:
    Entrada: [B, NZ, 1, 1]
    Camada 1: [B, NZ, 1, 1] -> [B, NGF*4, 4, 4]  (4x4)
    Camada 2: [B, NGF*4, 4, 4] -> [B, NGF*2, 8, 8]  (8x8)
    Camada 3: [B, NGF*2, 8, 8] -> [B, NGF, 16, 16] (16x16)
    Camada 4: [B, NGF, 16, 16] -> [B, NC, 32, 32]   (32x32)
    """
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            # Bloco 1: Projeção inicial de 1x1 para 4x4
            nn.ConvTranspose2d(nz, ngf * 4, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # Bloco 2: Upsampling de 4x4 para 8x8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # Bloco 3: Upsampling de 8x8 para 16x16
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # Bloco 4: Upsampling final de 16x16 para 32x32
            nn.ConvTranspose2d(ngf, nc, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)


class Discriminator(nn.Module):
    """
    Discriminador DCGAN para imagens 32x32:
    Entrada: [B, NC, 32, 32]
    Camada 1: [B, NC, 32, 32] -> [B, NDF, 16, 16] (Downsample 16x16, sem BN)
    Camada 2: [B, NDF, 16, 16] -> [B, NDF*2, 8, 8]  (Downsample 8x8)
    Camada 3: [B, NDF*2, 8, 8] -> [B, NDF*4, 4, 4]  (Downsample 4x4)
    Camada 4: [B, NDF*4, 4, 4] -> [B, 1, 1, 1]      (Downsample 1x1)
    """
    def __init__(self, ndf=NDF, nc=NC):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            # Bloco 1: Convolução de entrada (sem BatchNorm)
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloco 2: Downsampling para 8x8
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloco 3: Downsampling para 4x4
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloco 4: Convolução final projetando em escalar 1x1
            nn.Conv2d(ndf * 4, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)


# Inicialização de pesos de Radford et al.
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)



In [ ]:
# Instanciar modelos, aplicar inicialização e enviar para GPU
netG = Generator().to(device)
netD = Discriminator().to(device)

netG.apply(weights_init)
netD.apply(weights_init)

# Contagem de Parâmetros
params_G = sum(p.numel() for p in netG.parameters() if p.requires_grad)
params_D = sum(p.numel() for p in netD.parameters() if p.requires_grad)

print(f"🧠 Gerador G:     {params_G:,} parâmetros treináveis")
print(f"🛡️ Discriminador D: {params_D:,} parâmetros treináveis")

# Teste de Sanidade de Formato Tensorial (Tensor Shape Tracing)
dummy_z = torch.randn(8, NZ, 1, 1, device=device)
dummy_fake = netG(dummy_z)
dummy_pred = netD(dummy_fake)

print(f"\n🧪 Sanity Check:")
print(f"  Vetor de Ruído z:        {list(dummy_z.shape)}")
print(f"  Imagem Gerada G(z):      {list(dummy_fake.shape)} (Valores: [{dummy_fake.min().item():.2f}, {dummy_fake.max().item():.2f}])")
print(f"  Probabilidade D(G(z)):   {list(dummy_pred.shape)}")
assert dummy_fake.shape == (8, 3, 32, 32), "Erro no shape gerado pelo Gerador!"
assert dummy_pred.shape == (8, 1), "Erro no shape predito pelo Discriminador!"
print("  ✅ Rastreamento de Tensores Validado com Sucesso!")



---
## ⚔️ Parte 4: A Dinâmica Minimax e a Heurística Não-Saturante

### A Derivada que Salva o Treinamento: $\log(1 - D)$ vs $-\log D$
No artigo original de Ian Goodfellow (2014), o objetivo teórico do jogo de soma zero é:
$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log (1 - D(G(z)))]$$

No início do treinamento, o gerador produz ruído puro. O discriminador aprende rapidamente a atribuir $D(G(z)) \approx 0$.
A derivada da perda teórica com relação à saída do discriminador é:
$$\frac{\partial}{\partial D} \log(1 - D) = -\frac{1}{1 - D}$$
Quando propagada pela ativação Sigmoid do discriminador ($D = \sigma(a)$), o gradiente com relação à ativação pré-sigmoid $a$ torna-se:
$$\frac{\partial \mathcal{L}}{\partial a} = D$$
**O Gargalo Prático:** Se $D \approx 0$, o gradiente é **zero**! O gerador satura e para de aprender no instante exato em que ele mais precisa de sinal!

**A Solução de Engenharia (Heurística Não-Saturante):**
Treinamos $G$ para maximizar $\log D(G(z))$, o que equivale a minimizar $-\log D(G(z))$.
Calculando a derivada correspondente:
$$\frac{\partial \mathcal{L}_{\text{NS}}}{\partial a} = -(1 - D)$$
Quando $D \approx 0$, o gradiente é proporcional a **$1.0$** — força máxima de correção!

### Desacoplamento de Grafos com `.detach()`:
Ao atualizar $D$, geramos imagens falsas $\tilde{x} = G(z)$. Se passássemos $\tilde{x}$ diretamente para $D$ sem desacoplamento, a chamada `loss_D.backward()` retropropagaria derivadas através de todo o gerador $G$, gastando o dobro de VRAM e gerando atualizações espúrias.
Chamando `D(fake_imgs.detach())`, cortamos o grafo computacional, tratando as imagens sintéticas como tensores constantes durante o passo de $D$.



In [ ]:
# Função de perda (Binary Cross-Entropy)
criterion = nn.BCELoss()

# Vetor de ruído latente fixo para monitorar o progresso visual de épocas em épocas
FIXED_NOISE = torch.randn(64, NZ, 1, 1, device=device)

# Hiperparâmetros de Otimização canônicos de Radford et al. (ICLR 2016)
LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999

optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, BETA2))

# Rótulos para BCE
REAL_LABEL = 0.9  # One-sided Label Smoothing (0.9 em vez de 1.0 para regularizar o Discriminador)
FAKE_LABEL = 0.0

print(f"🎯 Otimizador Adam: LR={LR}, Betas=({BETA1}, {BETA2})")
print(f"🛡️ Label Smoothing: Real={REAL_LABEL}, Fake={FAKE_LABEL}")



In [ ]:
# Número de épocas para treino
NUM_EPOCHS = 20

# Listas de telemetria
history = {
    'loss_D': [],
    'loss_G': [],
    'D_x': [],      # Média de D em imagens reais
    'D_G_z1': [],   # Média de D em fakes antes de atualizar G
    'D_G_z2': []    # Média de D em fakes após atualizar G
}

img_list = []

print(f"🚀 Iniciando Treinamento Adversarial da DCGAN ({NUM_EPOCHS} Épocas)...")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    loop = tqdm(dataloader, desc=f"Época [{epoch:02d}/{NUM_EPOCHS:02d}]", leave=True)
    
    epoch_loss_D = 0.0
    epoch_loss_G = 0.0
    epoch_D_x = 0.0
    epoch_D_G_z1 = 0.0
    epoch_D_G_z2 = 0.0
    
    for i, (real_imgs, _) in enumerate(loop):
        b_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)

        # =====================================================================
        # PASSO 1: Atualização do Discriminador D: maximizar log(D(x)) + log(1 - D(G(z)))
        # =====================================================================
        optimizerD.zero_grad()

        # 1.1 Lote com Imagens Reais
        labels_real = torch.full((b_size, 1), REAL_LABEL, dtype=torch.float, device=device)
        output_real = netD(real_imgs)
        loss_D_real = criterion(output_real, labels_real)
        loss_D_real.backward()
        d_x = output_real.mean().item()

        # 1.2 Lote com Imagens Falsas (com BARREIRA .detach() para isolar G)
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake_imgs = netG(noise)
        labels_fake = torch.full((b_size, 1), FAKE_LABEL, dtype=torch.float, device=device)
        output_fake = netD(fake_imgs.detach())
        loss_D_fake = criterion(output_fake, labels_fake)
        loss_D_fake.backward()
        d_g_z1 = output_fake.mean().item()

        loss_D = loss_D_real + loss_D_fake
        optimizerD.step()

        # =====================================================================
        # PASSO 2: Atualização do Gerador G: maximizar log(D(G(z))) [Heurística Não-Saturante]
        # =====================================================================
        optimizerG.zero_grad()
        
        # Passamos as imagens geradas SEM detach: os gradientes fluem através de D até G
        labels_g = torch.full((b_size, 1), 1.0, dtype=torch.float, device=device)  # Quer que D ache que são reais (1.0)
        output_g = netD(fake_imgs)
        loss_G = criterion(output_g, labels_g)
        loss_G.backward()
        d_g_z2 = output_g.mean().item()
        
        optimizerG.step()

        # Acumular telemetria
        epoch_loss_D += loss_D.item()
        epoch_loss_G += loss_G.item()
        epoch_D_x += d_x
        epoch_D_G_z1 += d_g_z1
        epoch_D_G_z2 += d_g_z2

        loop.set_postfix({
            'Loss_D': f"{loss_D.item():.3f}",
            'Loss_G': f"{loss_G.item():.3f}",
            'D(x)': f"{d_x:.2f}",
            'D(G(z))': f"{d_g_z2:.2f}"
        })

    # Médias da época
    n_batches = len(dataloader)
    history['loss_D'].append(epoch_loss_D / n_batches)
    history['loss_G'].append(epoch_loss_G / n_batches)
    history['D_x'].append(epoch_D_x / n_batches)
    history['D_G_z1'].append(epoch_D_G_z1 / n_batches)
    history['D_G_z2'].append(epoch_D_G_z2 / n_batches)

    # Gerar amostras com o vetor de ruído fixo para salvar o progresso da época
    with torch.no_grad():
        fake_fixed = netG(FIXED_NOISE).detach().cpu()
        img_list.append(vutils.make_grid(denormalize(fake_fixed[:32]), nrow=8, padding=2, normalize=False))

print("=" * 70)
print("🎉 Treinamento da DCGAN Concluído com Sucesso!")



---
## 📊 Parte 5: Diagnóstico de Treinamento e Curvas de Convergência

Analisamos o comportamento das perdas e das probabilidades médias do discriminador $D(x)$ e $D(G(z))$.
- Em um treinamento saudável, as perdas de $G$ e $D$ oscilam dinamicamente em torno de um equilíbrio, sem que uma das redes aniquile a outra.
- A probabilidade $D(x)$ deve permanecer alta ($> 0.6$) enquanto $D(G(z))$ sobe gradualmente de valores próximos a zero para a faixa de $0.3$ a $0.5$ (indicando que o gerador está progressivamente enganando o discriminador).



In [ ]:
plt.figure(figsize=(14, 5))

# Painel 1: Curvas de Perda
plt.subplot(1, 2, 1)
plt.plot(history['loss_D'], label='Loss Discriminador (D)', color='#E65100', lw=2)
plt.plot(history['loss_G'], label='Loss Gerador (G)', color='#0284C7', lw=2)
plt.title('Dinâmica de Perdas Adversariais (DCGAN)', fontsize=12, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss (BCELoss)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Painel 2: Probabilidades Médias D(x) vs D(G(z))
plt.subplot(1, 2, 2)
plt.plot(history['D_x'], label='D(x) [Imagens Reais]', color='#16A34A', lw=2)
plt.plot(history['D_G_z2'], label='D(G(z)) [Imagens Geradas]', color='#9333EA', lw=2)
plt.axhline(0.5, color='gray', linestyle=':', label='Equilíbrio Ideal (D = 0.5)')
plt.title('Telemetria de Probabilidades do Discriminador', fontsize=12, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Probabilidade Média')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
# Comparativo Visual: Início do Treinamento vs Época Final
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Amostras da Época 1 (ruído inicial)
axes[0].axis("off")
axes[0].set_title("Época 01: Ruído e Formas Incipientes", fontsize=11, fontweight='bold')
axes[0].imshow(np.transpose(img_list[0].numpy(), (1, 2, 0)))

# Amostras da Última Época (estruturas aprendidas)
axes[1].axis("off")
axes[1].set_title(f"Época {NUM_EPOCHS:02d}: Amostras Sintéticas Geradas", fontsize=11, fontweight='bold')
axes[1].imshow(np.transpose(img_list[-1].numpy(), (1, 2, 0)))

plt.tight_layout()
plt.show()



---
## 🌌 Parte 6: Interpolação no Espaço Latente ($z_A \to z_B$)

### Prova Geométrica de que a GAN Não Decorou o Dataset
Uma pergunta fundamental em modelagem generativa é: **o gerador aprendeu uma variedade contínua real ou ele apenas memorizou e reproduziu amostras de treino?**

Se o modelo tivesse memorizado os dados, transições no espaço latente resultariam em saltos abruptos e imagens intermediárias disformes.
Se o modelo mapeou com sucesso a variedade latente contínua, transitar suavemente entre dois vetores aleatórios $z_A$ e $z_B$:
$$z(\alpha) = (1 - \alpha) z_A + \alpha z_B, \quad \alpha \in [0, 1]$$
resultará em uma metamorfose visual contínua e coerente de cores, ângulos e morfologias.



In [ ]:
# Selecionar dois pontos latentes aleatórios
z_A = torch.randn(1, NZ, 1, 1, device=device)
z_B = torch.randn(1, NZ, 1, 1, device=device)

# Interpolação Linear esférica em 10 passos
STEPS = 10
alphas = np.linspace(0, 1, STEPS)
interpolated_z = []

for alpha in alphas:
    # Combinação linear ponderada
    z_interp = (1.0 - alpha) * z_A + alpha * z_B
    interpolated_z.append(z_interp)

interpolated_z = torch.cat(interpolated_z, dim=0)

# Gerar imagens ao longo da trajetória no espaço latente
netG.eval()
with torch.no_grad():
    interpolated_imgs = netG(interpolated_z)

# Plotar o trajeto visual contínuo
fig, axes = plt.subplots(1, STEPS, figsize=(18, 3))
for i in range(STEPS):
    img = denormalize(interpolated_imgs[i]).cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    axes[i].imshow(np.clip(img, 0, 1))
    axes[i].set_title(f"α = {alphas[i]:.2f}", fontsize=9)
    axes[i].axis('off')

plt.suptitle("Interpolação Suave no Espaço Latente: Metamorfose Contínua z_A ➔ z_B", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()



---
## 💡 Parte 7: Desafios Práticos e Conexão com o Próximo Caderno

### 🧠 Exercícios Recomendados para Pós-Graduação:
1. **Teste com Outras Classes do CIFAR-10:** Experimente alterar `FILTER_CLASS` para a classe `1` ('automobile') ou `7` ('horse') e observe a rapidez com que a DCGAN aprende texturas de lataria e silhuetas de animais.
2. **Impacto do Label Smoothing:** No discriminador, altere `REAL_LABEL` de $0.9$ para $1.0$ e observe se o discriminador começa a dominar excessivamente o gerador (causando gradientes menores para $G$).
3. **Exploração de Arquiteturas:** Adicione uma camada convolucional extra de resolução $64\times 64$ (`NGF/NDF`) e teste o comportamento com imagens redimensionadas para resolução superior.

---

### 🔗 Continuação no Caderno 2: GAN Condicional (cGAN) & Avaliação de Recall
Agora que dominamos a arquitetura pura e o loop de treino adversarial da DCGAN básica, avance para o **Caderno 2 (`aula_07_gans_generative_adversarial_networks.ipynb`)**, onde abordaremos:
- Injeção de condições/marcadores via embeddings no gerador e discriminador (**cGAN**).
- Resolução do desbalanceamento severo de classes em visão computacional biomédica (**Virtual Staining & Holo2Bright**).
- Demonstração prática do salto no **Recall (Sensibilidade)** da classe minoritária em conjunto de teste 100% real.

